# Run E2E (CORING) — Kaggle

Chay CA pipeline bang `scripts/run_e2e.sh`, override `METHOD=coring` inline: dense -> prune -> finetune lean -> benchmark.

**Truoc khi chay:** Accelerator=GPU; Add dataset: (a) folder code (co `prune.py` + `scripts/`), (b) data (`images/{train,val}`).

**Luu y:** e2e gom train dense (~70ep) + finetune (~80ep) -> co the LAU, coi chung gioi han gio session Kaggle.

In [ ]:
import os, sys, subprocess, glob, shutil

# deps: benchmark (thop) + coring (tensorly, tqdm)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "thop", "tensorly", "tqdm"], check=True)

# copy code (nhan dien bang prune.py) sang /kaggle/working de chay
hits = glob.glob("/kaggle/input/**/prune.py", recursive=True)
assert hits, "Khong thay code! Add dataset chua thu muc code (co prune.py)."
SRC  = os.path.dirname(hits[0])
CODE = "/kaggle/working/code"
if os.path.exists(CODE):
    shutil.rmtree(CODE)
shutil.copytree(SRC, CODE, ignore=shutil.ignore_patterns(
    "NewDeepfish", "CORING", ".git", ".cadence", "output", "*.pth", "__pycache__"))
os.chdir(CODE)
print("Code:", CODE, "| files:", sorted(os.listdir(CODE))[:12])

import torch
print("Torch:", torch.__version__, "| CUDA:", torch.cuda.is_available())
assert torch.cuda.is_available(), "Bat GPU o Settings > Accelerator!"

In [ ]:
cands = glob.glob("/kaggle/input/**/images/train", recursive=True)
assert cands, "Khong thay data! Can folder co images/{train,val} + labels/{train,val}."
DATA = os.path.dirname(os.path.dirname(cands[0]))
print("DATA =", DATA)

In [ ]:
# --- chay E2E: override flag INLINE truoc lenh (giong lenh chay tren server) ---
# them flag khac tuong tu: WANDB=1 TARGET_SPARSITY=0.7 BACKBONE=resnet50 ...
!METHOD=coring DATA={DATA} PY={sys.executable} bash scripts/run_e2e.sh